## Importing the libraries

In [ ]:
import pandas as pd
import numpy as np
import keras
from ucimlrepo import fetch_ucirepo

In [ ]:
# fetch dataset 
mushroom = fetch_ucirepo(id=73) 
  
# data (as pandas dataframes) 
X = mushroom.data.features 
y = mushroom.data.targets 
  
#Create the Dataframe
df = pd.DataFrame(X)
df['target'] = y

# metadata 
print(mushroom.metadata) 
  
# variable information 
print(mushroom.variables)

print(df.head())

## Data preprocessing

In [ ]:
columns = df.columns
col_hot = []
col_bin = []

for col in columns:
    print(col,df[col].unique())
    if df[col].nunique() > 2:
        col_hot.append(col)
    else:
        col_bin.append(col)


#print(col_hot)
#print(col_bin)

Refresh Import

In [ ]:
import importlib
import data_cleaning_and_preprocessing

importlib.reload(data_cleaning_and_preprocessing)

from data_cleaning_and_preprocessing import DataProcessor

# Create an instance of the DataProcessor class
processor = DataProcessor(df)

Handle Missing Data

In [ ]:
processor.handle_missing(strategy='most_frequent',columns=['stalk-root'])
processor.get_df().isnull().sum()

One-Hot and Binary Encoding

In [ ]:
col_bin = ['bruises','veil-type','target']
col_hot = columns.drop(col_bin).tolist()

print(col_hot)

# Preprocess the data
processor.encode_onehot(col_hot)

print(processor.get_df().head())


for col in col_bin:
    processor.map_binary(col)

In [ ]:
print(processor.get_df().head())

Variance-Threshold

In [ ]:
# d = processor.get_df()
# print(d.shape)
# print(d.dtypes.value_counts())
# print("num cols:", d.select_dtypes(include=np.number).shape[1])
# print("first cols:", d.columns[:10])


processor.variance_threshold()

processor.get_df().columns

print(processor.get_df().info())

Correlation Map

In [ ]:
# import matplotlib.pyplot as plt
# import seaborn as sns

# corr_matrix = processor.get_df().corr(numeric_only=True)
# print(corr_matrix.head())

# plt.figure(figsize=(12, 10))
# sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap='coolwarm')
# plt.title("Correlation Heatmap")
# plt.show()

## Split

In [ ]:
from sklearn.model_selection import train_test_split

X = processor.get_df().drop(columns=['target'])
y = processor.get_df()['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)



## Model Training

In [ ]:
# fetch dataset 
mushroom = fetch_ucirepo(id=73) 
  
# data (as pandas dataframes) 
X = mushroom.data.features 
y = mushroom.data.targets 

In [ ]:
print(X)

In [ ]:
import importlib
import wrapper

importlib.reload(wrapper)

from wrapper import DataProcessor

XGBoost

In [ ]:
#Command for installing xgboost
# import sys
# !{sys.executable} -m pip install -U xgboost

import xgboost as xgb

xgb_clf = xgb.XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    tree_method="hist",
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.9,
    colsample_bytree=0.9,
    random_state=42,
)

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Ridge
from sklearn.metrics import accuracy_score, classification_report
from wrapper import DataProcessorTransformer

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Ridge
from sklearn.metrics import accuracy_score, classification_report
from wrapper import DataProcessorTransformer

# X: dataframe features, y: target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# esempio mushroom: 'e' edible, 'p' poisonous
y_train = (y_train.squeeze() == "p").astype(int)
y_test = (y_test.squeeze()  == "p").astype(int)

col_bin = ["bruises", "veil-type"]
col_hot = [c for c in X.columns if c not in col_bin]

xgb_pipe = Pipeline(steps=[
    ("prep", DataProcessorTransformer(
        missing_strategy="most_frequent",
        missing_columns=["stalk-root"],
        binary_cols=col_bin,
        onehot_cols=col_hot,
        onehot_drop="first",
        # scegli quali colonne scalare (tipicamente numeriche, non one-hot)
        #scale_cols=[c for c in X.columns if c not in col_bin and c not in ["stalk-root"]],  # ADATTA
        #scale_method="standard",
        variance_threshold=0.0
    )),
    ("model", xgb_clf )
])

xgb_pipe.fit(X_train, y_train)
y_pred = xgb_pipe.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Logistic Regression

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

lr_pipe = Pipeline(steps=[
    ("prep", DataProcessorTransformer(
        missing_strategy="most_frequent",
        missing_columns=["stalk-root"],
        binary_cols=col_bin,
        onehot_cols=col_hot,
        onehot_drop="first",
        # scegli quali colonne scalare (tipicamente numeriche, non one-hot)
        #scale_cols=[c for c in X.columns if c not in col_bin and c not in ["stalk-root"]],  # ADATTA
        #scale_method="standard",
        variance_threshold=0.0
    )),
    ("model", LogisticRegression(max_iter=5000, solver="liblinear"))
])

lr_pipe.fit(X_train, y_train)

y_pred = lr_pipe.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Neural-Network

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Input, Dropout
from tensorflow.keras.callbacks import EarlyStopping

from scikeras.wrappers import KerasClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix, classification_report
from nn_model import build_model
# 1) build function (SciKeras la chiamerà in fit)
# def build_model(n_features):
#     model = Sequential([
#         Input(shape=(n_features,)),
#         Dense(32, activation="relu"),
#         Dropout(0.2),
#         Dense(16, activation="relu"),
#         Dropout(0.2),
#         Dense(1, activation="sigmoid")
#     ])
#     model.compile(
#         optimizer="adam",
#         loss="binary_crossentropy",
#         metrics=["accuracy", tf.keras.metrics.AUC(name="auc")]
#     )
#     return model

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=10,
    min_delta=1e-4,
    restore_best_weights=True
)

# 2) SciKeras estimator
# Important: n_features must be known after preprocessing.
# Easiest: let SciKeras infer input shape at runtime by using meta info:
# We'll pass n_features via model__n_features after we know it.
keras_clf = KerasClassifier(
    model=build_model,
    model__n_features=1,          # placeholder, will be overwritten after a small fit or using a trick below
    epochs=200,
    batch_size=64,
    verbose=2,
    callbacks=[early_stop],
    validation_split=0.2,         # easiest with pipeline (no need validation_data)
)

nn_pipe = Pipeline(steps=[
    ("prep", DataProcessorTransformer(
        missing_strategy="most_frequent",
        missing_columns=["stalk-root"],
        binary_cols=col_bin,
        onehot_cols=col_hot,
        onehot_drop="first",
        variance_threshold=0.0
    )),
    ("model", keras_clf)
])

# ---- trick to set correct n_features automatically ----
# Fit preprocessing alone to get transformed shape
Xtr_p = nn_pipe.named_steps["prep"].fit_transform(X_train, y_train)
n_features = Xtr_p.shape[1]
nn_pipe.named_steps["model"].set_params(model__n_features=n_features)

# Now fit full pipeline (preprocess will refit; ok, or you can reuse the fitted prep separately)
history = nn_pipe.fit(X_train, y_train)

# Predictions
y_proba = nn_pipe.predict_proba(X_test)[:, 1]
y_pred = (y_proba >= 0.5).astype(int)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_proba))
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

## Model saving

In [ ]:
import joblib
import os

os.makedirs("models", exist_ok=True)

joblib.dump(xgb_pipe, "models/xgb_model_pipe.joblib")
joblib.dump(lr_pipe, "models/lr_model_pipe.joblib")
joblib.dump(nn_pipe, "models/nn_model_pipe.joblib")
joblib.dump(columns, "models/columns.joblib")
joblib.dump(col_bin, "models/col_bin.joblib")
joblib.dump(col_hot, "models/col_hot.joblib")